# PasteTrace — Mamba Behavioral Sequence Model

**Yêu cầu:** Vào `Runtime → Change runtime type → T4 GPU` trước khi chạy.

## Pipeline
```
Bước 0  Setup (cài thư viện, clone repo, upload data)
Bước 1  Build Sequences  (meta.json → chuỗi sự kiện)
Bước 2  Make Splits      (chia train/val/test stratified)
Bước 3  Train            (Mamba model, lưu mamba.pt)
Bước 4  Test             (đánh giá trên test set, chạy 1 lần duy nhất)
Bước 5  Lưu model        (lưu lên Drive hoặc download về máy)
```

## Bước 0a — Kiểm tra GPU

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    raise RuntimeError('GPU chua bat! Vao Runtime → Change runtime type → T4 GPU roi chay lai.')

## Bước 0b — Cài thư viện Mamba (~3 phút lần đầu)

In [ ]:
# PyTorch + CUDA da co san tren Colab, chi can cai them:
!pip install mamba-ssm causal-conv1d --quiet
!pip install scikit-learn pandas plotly --quiet
print('Done!')

## Bước 0c — Clone repo từ GitHub

Lần đầu: clone repo về. Lần sau (runtime mới): clone lại hoặc pull update.

In [ ]:
import os

REPO_URL  = 'https://github.com/lequocviet-3103/Fraud-Detection.git'
REPO_DIR  = '/content/Fraud-Detection'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print('Repo da co, pull update...')
    !git -C {REPO_DIR} pull

os.chdir(REPO_DIR)
import sys; sys.path.insert(0, REPO_DIR)
print('Working dir:', os.getcwd())
!ls

## Bước 0d — Upload data PasteTrace lên Google Drive

**Làm 1 lần duy nhất** — data sẽ nằm trên Drive, mọi session sau dùng lại được.

**Trên máy tính Windows:**
```
Mở Google Drive trên trình duyệt
Tạo thư mục: My Drive / PasteTrace / data
Upload thư mục PasteTrace-release/ vào đó
```

Cấu trúc cần có trong Drive:
```
My Drive/
  PasteTrace/
    data/
      PasteTrace-release/
        PasteTrace-release/
          case studies/
            sp2023/
              pre-processed/
                111/ (có agrigation.csv)
                211/ (có agrigation.csv)
```

In [ ]:
# Mount Google Drive
from google.colab import drive
import os

drive.mount('/content/drive')

# Đường dẫn data trên Drive của bạn
DRIVE_DATA = '/content/drive/MyDrive/PasteTrace/data/PasteTrace-release'

if os.path.exists(DRIVE_DATA):
    # Tạo symlink vào project (không copy, tiết kiệm thời gian)
    link_dst = os.path.join(os.getcwd(), 'PasteTrace-release')
    if not os.path.exists(link_dst):
        os.symlink(DRIVE_DATA, link_dst)
    print('OK — Data linked từ Drive:', link_dst)
else:
    print(f'CANH BAO: Khong tim thay {DRIVE_DATA}')
    print('Kiem tra lai ten thu muc tren Drive.')

In [ ]:
# Kiểm tra data OK chưa
import os
DATA_ROOT = os.path.join(
    'PasteTrace-release', 'PasteTrace-release',
    'case studies', 'sp2023', 'pre-processed'
)
if os.path.isdir(DATA_ROOT):
    cases = [d for d in os.listdir(DATA_ROOT)
             if os.path.isdir(os.path.join(DATA_ROOT, d))]
    print(f'OK — {len(cases)} case folders: {cases}')
    for c in cases:
        students = os.listdir(os.path.join(DATA_ROOT, c))
        print(f'  {c}: {students}')
else:
    print('CANH BAO: Khong tim thay data root!')
    print('Path can co:', DATA_ROOT)

## Bước 1 — Build Sequences

Đọc meta.json → vector sự kiện T/P/C → `data/sequences/`

In [ ]:
!python -m src.data.build_sequences --min-events 3

In [ ]:
import pandas as pd
df = pd.read_csv('data/sequences_index.csv')
print(df[['id','label','n_events','time_available']].to_string())
print(f'\nTong: {len(df)} sinh vien | cheat={sum(df.label==1)} | normal={sum(df.label==0)}')
print(f'Events: min={df.n_events.min()}  median={df.n_events.median():.0f}  max={df.n_events.max()}')

## Bước 2 — Make Splits

⚠️ **sp2023 có 17 mẫu (15 cheat / 2 normal)** — rất ít cho train/val/test.
Nếu thấy cảnh báo `Cannot stratify` → **dùng LOO** ở bước 3.

In [ ]:
!python -m src.data.make_splits --train 0.7 --val 0.15 --test 0.15 --seed 42

In [ ]:
import json
with open('data/splits.json') as f:
    sp = json.load(f)
for s in ['train','val','test']:
    c = sp['counts'][s]
    print(f"{s:6}: {c['total']:2d} samples  (cheat={c['cheat']}, normal={c['normal']})  ids={sp[s]}")

## Bước 3 — Train

**Chọn 1 trong 2 cell:**
- **Cell A** — Train/val/test thông thường (khi splits có đủ mẫu)
- **Cell B** — LOO mode (**khuyến nghị** cho sp2023 17 mẫu)

In [ ]:
# Cell A: Train/Val/Test (chỉ dùng khi mỗi split có >= 1 mẫu mỗi class)
!python -m src.models.mamba_model train \
    --d-model 64 --n-layers 2 --dropout 0.2 \
    --epochs 80 --lr 1e-3 --patience 10 \
    --batch-size 8 --max-len 1000

In [ ]:
# Cell B: LOO (Leave-One-Out) — phù hợp cho dataset nhỏ
# Train lại 17 lần, mỗi lần bỏ 1 sinh viên ra test
!python -m src.models.mamba_model train --loo \
    --d-model 64 --n-layers 2 \
    --epochs 50 --lr 1e-3 --batch-size 4

In [ ]:
# Kiểm tra model files
import os
for fname in ['mamba.pt', 'config.json', 'scaler.json']:
    p = f'models/mamba/{fname}'
    if os.path.isfile(p):
        print(f'OK     {p}  ({os.path.getsize(p)/1024:.1f} KB)')
    else:
        print(f'MISSING  {p}')

## Bước 4 — Test (chỉ chạy 1 lần cuối cùng)

⚠️ Đây là **held-out test set** — không dùng để điều chỉnh hyperparameter.

In [ ]:
!python -m src.models.mamba_model test

In [ ]:
# Hiển thị kết quả
import json, pandas as pd
with open('results/mamba_metrics.json') as f:
    res = json.load(f)

m, mb = res['mamba'], res['majority_baseline']
rows = []
for name, mt in [('Mamba', m), ('Majority Baseline', mb)]:
    rows.append({'Model': name,
        'Accuracy':      f"{mt['accuracy']:.3f}",
        'Macro F1':      f"{mt['macro_f1']:.3f}",
        'Cheat F1':      f"{mt['cheat_f1']:.3f}",
        'Normal F1':     f"{mt['normal_f1']:.3f}",
        'Cheat Recall':  f"{mt['cheat_recall']:.3f}",
        'Normal Recall': f"{mt['normal_recall']:.3f}",
    })
print(pd.DataFrame(rows).to_string(index=False))

In [ ]:
# Confusion matrix
import matplotlib.pyplot as plt, numpy as np
cm = np.array(m['confusion'])
fig, ax = plt.subplots(figsize=(5,4))
ax.imshow(cm, cmap='Blues')
ax.set_xticks([0,1]); ax.set_xticklabels(['Pred Normal','Pred Cheat'])
ax.set_yticks([0,1]); ax.set_yticklabels(['True Normal','True Cheat'])
for i in range(2):
    for j in range(2):
        ax.text(j, i, str(cm[i,j]), ha='center', va='center',
                color='white' if cm[i,j] > cm.max()/2 else 'black', fontsize=18)
ax.set_title(f'Accuracy={m["accuracy"]:.3f}  Macro F1={m["macro_f1"]:.3f}')
plt.tight_layout(); plt.show()

## Bước 5 — Lưu model lên Google Drive

In [ ]:
# Lưu model lên Drive để dùng lại
import shutil, os

SAVE_DIR = '/content/drive/MyDrive/PasteTrace/trained_models/mamba'
os.makedirs(SAVE_DIR, exist_ok=True)

for src in ['models/mamba/mamba.pt', 'models/mamba/config.json',
            'models/mamba/scaler.json', 'results/mamba_metrics.json']:
    if os.path.isfile(src):
        shutil.copy2(src, SAVE_DIR)
        print(f'Saved {os.path.basename(src)} -> Drive')

print(f'\nDone! Model o Drive: {SAVE_DIR}')

In [ ]:
# (Tùy chọn) Download về máy tính
import zipfile
from google.colab import files

with zipfile.ZipFile('/content/mamba_trained.zip', 'w') as z:
    for f in ['models/mamba/mamba.pt', 'models/mamba/config.json',
              'models/mamba/scaler.json', 'results/mamba_metrics.json']:
        if os.path.isfile(f):
            z.write(f)
            print(f'Added {f}')

files.download('/content/mamba_trained.zip')

---
## Lần sau — Load model từ Drive (bỏ qua bước Train)

Khi mở Colab mới, chạy lại Bước 0a→0c→0d, rồi chạy cell này:

In [ ]:
import shutil, os
DRIVE_MODEL = '/content/drive/MyDrive/PasteTrace/trained_models/mamba'
os.makedirs('models/mamba', exist_ok=True)
for fname in ['mamba.pt', 'config.json', 'scaler.json']:
    shutil.copy2(f'{DRIVE_MODEL}/{fname}', f'models/mamba/{fname}')
    print(f'Loaded {fname} from Drive')

# Predict sinh viên mới
STUDENT_FOLDER = 'path/to/student_folder_with_meta_json'
!python -m src.models.mamba_model predict {STUDENT_FOLDER}

---
## Lỗi thường gặp

| Lỗi | Fix |
|-----|-----|
| `RuntimeError: GPU chua bat` | Runtime → Change runtime type → T4 GPU |
| `mamba_ssm not found` | Chạy lại Bước 0b |
| `sequences_index.csv not found` | Chạy Bước 1 |
| `Cannot stratify split` | Dùng Cell B (--loo) ở Bước 3 |
| `No meta.json found` | Kiểm tra DRIVE_DATA_PATH |
| Session bị reset sau 12h | Mount Drive lại, load model từ Drive |